# UEMR 2 trên YouCook2: căn chỉnh vector đoạn của Athena vào không gian Omni ft (A-Z)

Đặc tả: `docs/uemr2/UEMR_ALIGN_FINAL.md` (2026-10-23). Việc, thứ tự: `docs/uemr2/UEMR2_AGENT_AZ.md` mục 3 (số cell = số ở đó).
Mã: `uemr2/` (`uemr2_core.py` công thức, `data.py` cache → mảng, `athena_vec.py` q / u, `folds.py` cross-fitting).

**Hai lượt.** `PHASE = 'dev'`: cell 0–11 (train, chọn β trên dev, in khối protocol). Agent ghi `configs/uemr2_protocol.json`, commit.
`PHASE = 'val'`: ghim `FUSHION_REF` + `PROTO_SHA16`, chạy lại từ đầu (q, u, P_φ, h̄ đọc từ cache Drive; cell 11 chỉ kiểm lại) rồi 12–16.
Val không được chấm trước khi protocol được commit (cell 12 chặn).

Runtime: A100 (cell 5 train 2 fold Athena ≈ 2 × 18 phút; còn lại vài phút). Mọi cell chạy lại được sau khi bị ngắt.

In [1]:
# ===== 0. Lượt, cờ, đường dẫn Drive =====
from google.colab import drive
drive.mount('/content/drive')
import os, sys, json, glob, hashlib, shutil, subprocess, platform, time
T_START = time.time()

PHASE       = 'dev'      # 'dev' | 'val'
FUSHION_REF = 'chien'    # lượt val: commit ghi configs/uemr2_protocol.json
PROTO_SHA16 = None       # lượt val: sha256[:16] của configs/uemr2_protocol.json (bước C in ra)
ATHENA_REF  = 'chien'    # cần commit cd11ee8 (2 file fold) trở về sau
RUNS, SEEDS = ('U', 'A1', 'A2', 'A3', 'A4'), (0, 1, 2)
GRID        = [0, 0.1, 0.2, 0.3, 0.5, 0.75, 1.0]           # đặc tả 5.4, cho β (U) và β_I (e)
assert PHASE in ('dev', 'val')
assert PHASE == 'dev' or (PROTO_SHA16 and FUSHION_REF != 'chien'), 'lượt val: ghim FUSHION_REF (commit protocol) và PROTO_SHA16'

DRIVE  = '/content/drive/MyDrive/Colab Notebooks/code KL'
BASE   = f'{DRIVE}/Fushion/uemr'                    # UEMR: chỉ đọc
U2     = f'{DRIVE}/Fushion/uemr_2'                  # UEMR 2: mọi đầu ra
O, OT, UC = f'{BASE}/omni_cache', f'{BASE}/omni_cache_train', f'{BASE}/uniav_cache'
H8     = 'a793d2c4'                                  # vân tay đầu ra UniAV của Athena sạch (az_state.json 'new')
DBH, DBBH, DBTH = f'{O}/ft/{H8}', f'{O}/ft/{H8}_bo_sung_val', f'{OT}/ft/{H8}'
SUBSET = f'{O}/common/subset_videos.json'
META   = f'{DRIVE}/Omni/data/YouCookII/metadata'
TRAIN_MANIFEST     = f'{META}/train_omni_video.trainsplit.jsonl'
TRAIN_MANIFEST_SHA = 'e7ad8aee9a2d17e2845b73ada79930323ed4445d6d3ebe57a14a573b6d0b257b'   # bản sinh 2 fold (cd11ee8)
ATHENA_CKPT = f'{BASE}/athena_sach/961f1a89/athena_yc2clean.pth'
ATHENA_SHA  = '961f1a895f4b43c811d3e0cfa31448641989a0a0e39e421ed397682407643d58'
ATHENA_H8, ATHENA_RUN = ATHENA_SHA[:8], 'yc2clean_E1'
CAP_IV2, CAP_CHECK = f'{BASE}/athena_sach/caption_emb_iv2j_yc2clean.npz', f'{BASE}/athena_sach/captions_check.json'
RANKS_VAL = f'{BASE}/results/961f1a89/ranks_val394.npz'
TRAINED_LIST = f'{BASE}/athena_sach/961f1a89/train_videos.txt'   # 1.002 video 961f1a89 đã train (athena_sach ô 4, 7)
R6_TRAIN_SRC, R6_GAL_SRC = f'{UC}/athena_961f1a89_train', f'{UC}/athena_961f1a89'
LOCAL  = '/content/u2'
RES    = f'{U2}/results/{ATHENA_H8}'
for p in (DBH, DBBH, DBTH, SUBSET, TRAIN_MANIFEST, ATHENA_CKPT, CAP_IV2, CAP_CHECK, TRAINED_LIST, RANKS_VAL, R6_TRAIN_SRC, R6_GAL_SRC):
    print('%-6s %s' % ('OK' if os.path.exists(p) else 'THIẾU', p))
    assert os.path.exists(p), p
for d in (U2, RES, LOCAL):
    os.makedirs(d, exist_ok=True)

def sha256(path, bs=1 << 24):
    h = hashlib.sha256()
    with open(path, 'rb') as f:
        for chunk in iter(lambda: f.read(bs), b''):
            h.update(chunk)
    return h.hexdigest()

def sh(cmd, cwd=None, check=True):
    r = subprocess.run(cmd, shell=True, cwd=cwd, capture_output=True, text=True)
    if check and r.returncode:
        raise RuntimeError(f'lệnh lỗi ({r.returncode}): {cmd}\n{(r.stdout + r.stderr)[-3000:]}')
    return r.stdout.strip()

CHECK_PATH = f'{U2}/checks_{PHASE}.json'
CHECK = json.load(open(CHECK_PATH)) if os.path.exists(CHECK_PATH) else {}
def check(name, value):
    """Ghi một kiểm tra (đặc tả 7.4) vào checks_<PHASE>.json; cell 15 đọc lại cho tiêu chí 6."""
    CHECK[name] = value
    json.dump(CHECK, open(CHECK_PATH, 'w'), indent=1, ensure_ascii=False, default=str)
print('PHASE', PHASE, '| FUSHION_REF', FUSHION_REF, '| ATHENA_REF', ATHENA_REF)
assert os.path.isdir(U2)

Mounted at /content/drive
OK     /content/drive/MyDrive/Colab Notebooks/code KL/Fushion/uemr/omni_cache/ft/a793d2c4
OK     /content/drive/MyDrive/Colab Notebooks/code KL/Fushion/uemr/omni_cache/ft/a793d2c4_bo_sung_val
OK     /content/drive/MyDrive/Colab Notebooks/code KL/Fushion/uemr/omni_cache_train/ft/a793d2c4
OK     /content/drive/MyDrive/Colab Notebooks/code KL/Fushion/uemr/omni_cache/common/subset_videos.json
OK     /content/drive/MyDrive/Colab Notebooks/code KL/Omni/data/YouCookII/metadata/train_omni_video.trainsplit.jsonl
OK     /content/drive/MyDrive/Colab Notebooks/code KL/Fushion/uemr/athena_sach/961f1a89/athena_yc2clean.pth
OK     /content/drive/MyDrive/Colab Notebooks/code KL/Fushion/uemr/athena_sach/caption_emb_iv2j_yc2clean.npz
OK     /content/drive/MyDrive/Colab Notebooks/code KL/Fushion/uemr/athena_sach/captions_check.json
OK     /content/drive/MyDrive/Colab Notebooks/code KL/Fushion/uemr/athena_sach/961f1a89/train_videos.txt
OK     /content/drive/MyDrive/Colab Notebook

In [2]:
# ===== 1. Code Fushion + UniAV-fixed (nhánh chien), cài đặt, môi trường, unit test =====
FUSHION, ATHENA = '/content/code/Fushion_UEMR', '/content/code/UniAV-fixed'
os.makedirs('/content/code', exist_ok=True)
if not os.path.isdir(f'{FUSHION}/.git'):
    sh(f'git clone -q https://github.com/chien24/Fushion_UEMR.git {FUSHION}')
if not os.path.isdir(f'{ATHENA}/.git'):
    sh(f'git clone -q -b chien https://github.com/im-xiaoming/UniAV-fixed.git {ATHENA}')

def checkout(repo, ref):
    sh('git fetch -q origin', repo)
    branch = subprocess.run(['git', '-C', repo, 'rev-parse', '--verify', '-q', f'origin/{ref}'], capture_output=True).returncode == 0
    sh(f'git checkout -q --detach {"origin/" + ref if branch else ref}', repo)
    return sh('git rev-parse HEAD', repo)

FUSHION_COMMIT, ATHENA_COMMIT = checkout(FUSHION, FUSHION_REF), checkout(ATHENA, ATHENA_REF)
sh('git merge-base --is-ancestor cd11ee8 HEAD', ATHENA)          # có 2 fold
for f in ('configs/youcook2_event_yc2clean_foldA.yaml', 'configs/youcook2_event_yc2clean_foldB.yaml'):
    assert os.path.exists(f'{ATHENA}/{f}'), f
if FUSHION not in sys.path:
    sys.path.insert(0, FUSHION)
os.environ['PYTHONPATH'] = FUSHION + ':' + os.environ.get('PYTHONPATH', '')
os.chdir(FUSHION)
sh(f'{sys.executable} -m pip install -q --only-binary numpy -r requirements_uniav.txt')
import numpy as np, pandas as pd, torch
from IPython.display import display
gpu = sh('nvidia-smi --query-gpu=name,memory.total --format=csv,noheader', check=False)
ENV = {'python': platform.python_version(), 'numpy': np.__version__, 'pandas': pd.__version__, 'torch': torch.__version__,
       'cuda': torch.version.cuda, 'gpu': gpu or 'không có',
       'fushion': f'{FUSHION_REF} @ {FUSHION_COMMIT[:7]}', 'athena': f'{ATHENA_REF} @ {ATHENA_COMMIT[:7]}',
       'fushion dirty': sh('git status --porcelain --untracked-files=no', FUSHION) or 'không',
       'athena dirty': sh('git status --porcelain --untracked-files=no', ATHENA) or 'không'}
print(json.dumps(ENV, indent=1, ensure_ascii=False))
r = subprocess.run([sys.executable, '-m', 'pytest', 'uemr2/tests', 'omni_retrieval/tests', '-q', '-p', 'no:cacheprovider'],
                   cwd=FUSHION, capture_output=True, text=True)
print(r.stdout[-600:])
assert r.returncode == 0, 'unit test chưa qua: dừng'
assert ENV['fushion dirty'] == ENV['athena dirty'] == 'không', 'repo bị sửa'
assert torch.cuda.is_available(), 'cần GPU'
assert int(np.__version__.split('.')[0]) >= 2, 'checkpoint Athena cần numpy 2.x: Runtime > Restart session rồi chạy lại ô 0-1'

{
 "python": "3.13.16",
 "numpy": "2.1.3",
 "pandas": "2.2.3",
 "torch": "2.11.0+cu130",
 "cuda": "13.0",
 "gpu": "NVIDIA A100-SXM4-40GB, 40960 MiB",
 "fushion": "chien @ bde1ed2",
 "athena": "chien @ cd11ee8",
 "fushion dirty": "không",
 "athena dirty": "không"
}
........................................................................ [ 87%]
..........                                                               [100%]
82 passed in 13.04s



## Cột (b) và dữ liệu (đặc tả 7.4.1, 7.4.5)

In [3]:
# ===== 2. Nạp cache UEMR qua uemr2/data.py; tái lập cột (b) trên dev, val, kho 498 =====
from omni_retrieval.collect import local_npz
from omni_retrieval.evaluate import gallery_videos, load_text, rank_metrics, paired_bootstrap as omni_bootstrap
from uemr2 import data as D, uemr2_core as U
t0 = time.time()
DB  = str(local_npz(DBH, f'{LOCAL}/db_ft', ('segments.npz', 'text.npz', 'text_dev.npz'),
                    [p for r, p in D.ROWS.items() if r != 'R8']))
DBB = str(local_npz(DBBH, f'{LOCAL}/db_bs', ('segments.npz',), ('r8',)))
DBOF = lambda row: DBB if row == 'R8' else DB
VIDS = {'val': gallery_videos(SUBSET, 'val'), 'dev': gallery_videos(SUBSET, 'dev'), 'all': None}
TXT = {'val': load_text(DB), 'dev': load_text(DB, 'text_dev.npz')}
TXT['all'] = TXT['val']
Q = {s: D.queries(DB, 'text_dev.npz' if s == 'dev' else 'text.npz') for s in ('val', 'dev', 'all')}
assert (len(VIDS['val']), len(VIDS['dev']), len(Q['val']['T']), len(Q['dev']['T'])) == (394, 104, 3030, 773)
ROWS_OF = {'val': list(D.ROWS), 'dev': list(D.ROWS), 'all': ['R6', 'R2', 'R8', 'R6∪R2']}

def metrics(r):
    """R@k, MdR, MnR, VCMR@1 của một kết quả uemr2_core.evaluate (hoặc của reference)."""
    rank = r['rank']
    hit = r['vcmr'] if 'vcmr' in r else (rank <= 1) & r['moment_hit']
    m = rank_metrics(rank)
    return {'R@1': m['R@1'], 'R@5': m['R@5'], 'R@10': m['R@10'], 'MdR': m['MedR'], 'MnR': m['MnR'], 'VCMR@1': 100 * float(hit.mean())}

# (1) thứ hạng UEMR bằng chính code UEMR (load_index + video_ranks), so với số đã log
REFR = {(s, row): D.reference(DBOF(row), D.ROWS[row], TXT[s], VIDS[s]) for s in ROWS_OF for row in ROWS_OF[s]}
tab_b = pd.DataFrame([{'tập': s, 'dòng': row, **metrics(r), 'vec/video': r['vec/video']} for (s, row), r in REFR.items()])
display(tab_b.round(2))
bad = []
for s, ref in D.REF.items():
    for row, want in ref.items():
        got = tab_b[(tab_b['tập'] == s) & (tab_b['dòng'] == row)].iloc[0]
        tol = 1e-6 if s == 'dev' else 0.005 + 1e-9               # dev: số đủ chữ số; val, kho 498: số 2 chữ số
        bad += [(s, row, k, round(float(got[k]), 4), v) for k, v in want.items() if abs(float(got[k]) - v) > tol]
assert not bad, f'DỪNG (7.4.1): không tái lập được cột (b) {bad}'
with np.load(RANKS_VAL) as z:                                   # thứ hạng từng câu đã lưu (R0-R7, val)
    assert np.array_equal(z['caption_id'], TXT['val']['caption_id'])
    diff = [row for row in D.ROWS if row != 'R8' and not np.array_equal(z[D.ROWS[row]], REFR['val', row]['rank'])]
assert not diff, f'DỪNG: thứ hạng khác ranks_val394.npz ở {diff}'

# (2) cùng thứ hạng qua uemr2/data.py + uemr2_core (MaxSim, hạng chặt, VCMR)
G = {(s, row): D.gallery(DBOF(row), D.ROWS[row], VIDS[s]) for s in ROWS_OF for row in ROWS_OF[s]}
for s in ROWS_OF:
    D.mark_fill(G[s, 'R8'], G[s, 'R6'])                         # cờ đoạn lấp cho C1; assert K_pred mỗi video
CV = {k: D.cap_vid(Q[k[0]], g) for k, g in G.items()}

def run(key, parts):
    g = G[key]
    return U.run(parts, g['seg_vid'], len(g['videos']), CV[key], Q[key[0]]['gt'], g['segs'])

SOM = {k: D.scores(Q[k[0]]['T'], g['E']) for k, g in G.items()}  # S^Ω (đặc tả 5.1), vector Omni thô như UEMR
same = {}
for k in G:
    r = run(k, [(1.0, SOM[k])])
    same[k] = bool(np.array_equal(r['rank'], REFR[k]['rank']) and np.array_equal((r['rank'] <= 1) & r['moment_hit'], REFR[k]['vcmr']))
assert all(same.values()), f'DỪNG: uemr2_core khác thứ hạng UEMR ở {[k for k, v in same.items() if not v]}'

# (3) bootstrap của uemr2_core = của báo cáo UEMR (R8 - R6 val: +2.87 [+1.63, +4.19])
r8, r6, vv = REFR['val', 'R8']['rank'], REFR['val', 'R6']['rank'], Q['val']['video']
b_core = U.paired_bootstrap(100.0 * (r8 <= 1), 100.0 * (r6 <= 1), vv)
b_omni = omni_bootstrap(r8, r6, 1, 1000, 0, 'video', vv)
print('R8 - R6 val:', {k: round(v, 3) for k, v in b_core.items()}, '| omni_retrieval:', round(b_omni['ci_low'], 3), round(b_omni['ci_high'], 3))
assert abs(b_core['lo'] - b_omni['ci_low']) < 1e-9 and abs(b_core['hi'] - b_omni['ci_high']) < 1e-9
assert (round(b_core['diff'], 2), round(b_core['lo'], 2), round(b_core['hi'], 2)) == (2.87, 1.63, 4.19)
check('7.4.1 cột (b) tái lập', {f'{s} {row}': round(float(tab_b[(tab_b["tập"] == s) & (tab_b["dòng"] == row)].iloc[0]["R@1"]), 2)
                                 for s in ROWS_OF for row in ROWS_OF[s]})
check('uemr2_core == UEMR (hạng + VCMR, mọi dòng, dev/val/498)', True)
print(f'cell 2: {time.time() - t0:.0f} s')

,tập,dòng,R@1,R@5,R@10,MdR,MnR,VCMR@1,vec/video
0,val,R0,8.32,26.93,38.75,18.5,49.19,0.03,1.00
1,val,R3,18.02,44.42,57.85,7.0,24.79,6.07,7.52
2,val,R6,24.52,51.72,64.69,5.0,22.23,15.97,7.52
3,val,R2,27.16,58.88,73.40,4.0,14.57,9.87,22.66
4,val,R8,27.39,56.86,70.86,4.0,15.52,14.32,22.15
5,val,R6∪R2,29.90,62.11,76.27,3.0,12.68,15.02,30.18
6,val,R7,37.52,69.80,80.96,2.0,9.00,35.64,7.69
7,dev,R0,23.54,52.91,66.11,5.0,13.19,0.00,1.00
8,dev,R3,37.00,69.99,81.37,2.0,7.20,9.70,6.99
9,dev,R6,43.86,74.90,85.64,2.0,6.31,27.17,6.99


R8 - R6 val: {'diff': 2.871, 'lo': 1.632, 'hi': 4.188} | omni_retrieval: 1.632 4.188
cell 2: 59 s


In [4]:
# ===== 3. Kiểm dữ liệu: dev, val không giao train; số đoạn, số câu từng tập; nguồn R6 train =====
from uemr2.folds import manifest_videos
DBT = str(local_npz(DBTH, f'{LOCAL}/db_train', ('segments.npz', 'text.npz'), ('gt', 'uniav_pred')))
TR, QT = D.train_rows(DBT), D.queries(DBT)
assert sha256(TRAIN_MANIFEST) == TRAIN_MANIFEST_SHA, 'manifest trainsplit khác bản đã sinh 2 fold'
M_TR = set(manifest_videos(TRAIN_MANIFEST))
TRAIN_VIDS = set(TR['video'].tolist())
GAL = VIDS['val'] | VIDS['dev']
DATA = {'train videos': len(TRAIN_VIDS), 'train == manifest trainsplit': TRAIN_VIDS == M_TR,
        'train ∩ (dev ∪ val)': len(TRAIN_VIDS & GAL), 'caption train ∩ (dev ∪ val)': len(set(QT['video'].tolist()) & GAL),
        'dev ∩ val': len(VIDS['dev'] & VIDS['val']), 'train GT rows': int((TR['partition'] == 'gt').sum()),
        'train R6 rows': int((TR['partition'] == 'uniav_pred').sum()), 'train captions': len(QT['T'])}
print(json.dumps(DATA, indent=1, ensure_ascii=False))
assert DATA['train videos'] == 1002 and DATA['train == manifest trainsplit']
assert DATA['train ∩ (dev ∪ val)'] == DATA['caption train ∩ (dev ∪ val)'] == DATA['dev ∩ val'] == 0
assert (DATA['train GT rows'], DATA['train R6 rows'], DATA['train captions']) == (7787, 7802, 7787)

# R6 train: state_dict_seg của 961f1a89, cùng θ và hậu xử lý với R6 val (B1 ô 9-10)
src_t, src_g = (json.load(open(f'{d}/SOURCE.json')) for d in (R6_TRAIN_SRC, R6_GAL_SRC))
R6SRC = {k: (src_t.get(k), src_g.get(k)) for k in ('sha256', 'seg_weights', 'theta', 'postprocess', 'feature_source', 'segments', 'videos')}
print(json.dumps(R6SRC, indent=1, ensure_ascii=False, default=str))
assert src_t['sha256'] == src_g['sha256'] == ATHENA_SHA and src_t['split'] == 'trainsplit'
assert src_t['seg_weights'].startswith('state_dict_seg') and src_t['seg_weights'] == src_g['seg_weights']
assert src_t['theta'] == src_g['theta'] and src_t['theta']['theta'] == 0.4 and src_t['postprocess'] == src_g['postprocess']
assert (src_t['videos'], src_t['segments']) == (1002, 7802)
per_jsonl = pd.Series([json.loads(l)['video_id'] for l in open(f'{R6_TRAIN_SRC}/segments_pred.jsonl') if l.strip()]).value_counts()
per_rows = pd.Series(TR['video'][TR['partition'] == 'uniav_pred']).value_counts()
assert per_jsonl.sort_index().equals(per_rows.sort_index()), 'số đoạn R6 train mỗi video khác segments_pred.jsonl'

rows = []
for (s, row), g in G.items():
    rows.append({'tập': s, 'dòng': row, 'video': len(g['videos']), 'đoạn': len(g['seg_key']), 'vec/video': len(g['seg_key']) / len(g['videos']),
                 'câu': len(Q[s]['T']), 'đoạn lấp': int(g['is_fill'].sum()) if 'is_fill' in g else ''})
display(pd.DataFrame(rows).round(2))
check('7.4.5 dữ liệu: dev, val không giao train', DATA)
check('nguồn R6 train', R6SRC)

{
 "train videos": 1002,
 "train == manifest trainsplit": true,
 "train ∩ (dev ∪ val)": 0,
 "caption train ∩ (dev ∪ val)": 0,
 "dev ∩ val": 0,
 "train GT rows": 7787,
 "train R6 rows": 7802,
 "train captions": 7787
}
{
 "sha256": [
  "961f1a895f4b43c811d3e0cfa31448641989a0a0e39e421ed397682407643d58",
  "961f1a895f4b43c811d3e0cfa31448641989a0a0e39e421ed397682407643d58"
 ],
 "seg_weights": [
  "state_dict_seg (best_seg, dev R@0.5)",
  "state_dict_seg (best_seg, dev R@0.5)"
 ],
 "theta": [
  {
   "theta": 0.4,
   "chosen_on": "dev",
   "rule": "dev: mean K_pred closest to mean K_GT, tie -> smaller theta",
   "grid": [
    0.05,
    0.1,
    0.15,
    0.2,
    0.25,
    0.3,
    0.35,
    0.4,
    0.45,
    0.5,
    0.55,
    0.6
   ],
   "videos": 104,
   "checkpoint_sha256": "961f1a895f4b43c811d3e0cfa31448641989a0a0e39e421ed397682407643d58",
   "theta_old_checkpoint": 0.4
  },
  {
   "theta": 0.4,
   "chosen_on": "dev",
   "rule": "dev: mean K_pred closest to mean K_GT, tie -> smaller th

,tập,dòng,video,đoạn,vec/video,câu,đoạn lấp
0,val,R0,394,394,1.00,3030,
1,val,R3,394,2962,7.52,3030,
2,val,R6,394,2962,7.52,3030,
3,val,R2,394,8929,22.66,3030,
4,val,R8,394,8728,22.15,3030,5766
5,val,R6∪R2,394,11891,30.18,3030,
6,val,R7,394,3030,7.69,3030,
7,dev,R0,104,104,1.00,773,
8,dev,R3,104,727,6.99,773,
9,dev,R6,104,727,6.99,773,


In [5]:
# ===== 4. Omni ft: đủ vector cho mọi đoạn và mọi câu =====
# shortcut: chỉ kiểm. Mục 1 (nhật ký) không thiếu gì; nếu ô này báo thiếu thì dừng, agent viết ô encode
# (omni_retrieval.encode + checkpoint-1948, khoá seg_key) cho đúng phần thiếu.
def part_file(name, train=False):
    roots = ([f'{OT}/common/partitions_{H8}', f'{OT}/common/partitions'] if train else
             [f'{O}/common/partitions_{H8}', f'{O}/common/partitions', f'{BASE}/results/{ATHENA_H8}/bo_sung/partitions_val'])
    for r in roots:
        if os.path.exists(f'{r}/{name}.json'):
            return f'{r}/{name}.json'
    raise FileNotFoundError(name)

COVER = []
for row, p in D.ROWS.items():
    need = sum(len(s) for v, s in json.load(open(part_file(p))).items() if v in GAL)
    have = len(G['val', row]['seg_key']) + len(G['dev', row]['seg_key'])
    COVER.append({'tập': 'val + dev', 'dòng': row, 'cần': need, 'có': have})
for p in ('gt', 'uniav_pred'):
    need = sum(len(s) for s in json.load(open(part_file(p, train=True))).values())
    COVER.append({'tập': 'train', 'dòng': p, 'cần': need, 'có': int((TR['partition'] == p).sum())})
n_cap_tr = sum(1 for l in open(TRAIN_MANIFEST) if l.strip())
COVER += [{'tập': 'câu val', 'dòng': 't', 'cần': 3030, 'có': len(Q['val']['T'])},
          {'tập': 'câu dev', 'dòng': 't', 'cần': 773, 'có': len(Q['dev']['T'])},
          {'tập': 'câu train', 'dòng': 't', 'cần': n_cap_tr, 'có': len(QT['T'])}]
cover = pd.DataFrame(COVER)
cover['thiếu'] = cover['cần'] - cover['có']
display(cover)
check('Omni ft đủ (cần = có)', bool((cover['thiếu'] == 0).all()))
assert (cover['thiếu'] == 0).all(), 'DỪNG: thiếu vector Omni ft, báo agent để viết ô encode phần thiếu'

,tập,dòng,cần,có,thiếu
0,val + dev,R0,498,498,0
1,val + dev,R3,3689,3689,0
2,val + dev,R6,3689,3689,0
3,val + dev,R2,11316,11316,0
4,val + dev,R8,11081,11081,0
5,val + dev,R6∪R2,15005,15005,0
6,val + dev,R7,3803,3803,0
7,train,gt,7787,7787,0
8,train,uniav_pred,7802,7802,0
9,câu val,t,3030,3030,0


## Athena: 2 fold, q, u (đặc tả 3.2, 3.3, 4.4; kiểm 7.4.3, 7.4.4)

In [6]:
# ===== 5. Fold: uemr2/folds.py, train Athena-A, Athena-B, export --no-seg; sha256 =====
from uemr2 import folds as Fo
FOLD_DIR = f'{U2}/athena_folds'
os.makedirs(FOLD_DIR, exist_ok=True)
t0 = time.time()
# (1) môi trường train của Athena (như notebooks/athena_sach_yc2_A.ipynb ô 2-3)
import site, importlib
sh(f'{sys.executable} -m pip install -q pycocoevalcap h5py tensorboard')
if subprocess.run([sys.executable, '-c', 'import torch, nms_1d_cpu'], capture_output=True).returncode:
    sh(f'{sys.executable} setup.py install --user > /content/nms_build.log 2>&1', f'{ATHENA}/libs/utils')
sh(f'{sys.executable} -c "import torch, nms_1d_cpu"')
if site.getusersitepackages() not in sys.path:                 # kernel này cũng import được nms_1d_cpu (cell 6)
    sys.path.append(site.getusersitepackages())
importlib.invalidate_caches()
from huggingface_hub import snapshot_download
FEATS, TEACHER = f'{ATHENA}/data/youcookii/iv2_feats', f'{ATHENA}/data/youcookii/omni_emb_full.npz'
if not (os.path.exists(f'{FEATS}/.done') and os.path.exists(TEACHER)):
    import tarfile
    d = snapshot_download('nguyenminh04/uniav-youcook2-data', repo_type='dataset', local_dir='/content/hf_yc',
                          allow_patterns=['iv2_feats/*', 'teacher/*'])
    os.makedirs(FEATS, exist_ok=True)
    for t in sorted(glob.glob(f'{d}/iv2_feats/*.tar')):
        with tarfile.open(t) as tf:
            tf.extractall(FEATS, filter='data')
    shutil.copy(f'{d}/teacher/omni_emb_full.npz', TEACHER)
    open(f'{FEATS}/.done', 'w').write('ok')
CAP_LOCAL = f'{ATHENA}/data/youcookii/caption_emb_iv2j_yc2clean.npz'
CAP_SHA = json.load(open(CAP_CHECK))['sha256']
if not (os.path.exists(CAP_LOCAL) and sha256(CAP_LOCAL) == CAP_SHA):
    shutil.copy(CAP_IV2, CAP_LOCAL)
assert sha256(CAP_LOCAL) == CAP_SHA and len(glob.glob(f'{FEATS}/*.npz')) >= 1500

# (2) sinh lại 2 fold từ manifest trên Drive (nguồn 1.002 video: train_omni_video.trainsplit.jsonl; subset_videos.json chỉ
#     để loại dev/val). Danh sách phải = tập Athena nạp khi train config yc2clean (= tập 961f1a89 đã train), và 2 fold phải
#     trùng từng byte với file đã commit ở repo Athena
EXCL = GAL | set(Fo.manifest_videos(f'{META}/val_omni_video.jsonl'))
ATH_TR = Fo.athena_train_videos(ATHENA, CAP_LOCAL)
TRAINED = {l.strip() for l in open(TRAINED_LIST) if l.strip()}
assert set(ATH_TR) == TRAINED, 'dataset Athena yc2clean khác train_videos.txt của 961f1a89'
FOLD = Fo.write_folds(ATHENA, sorted(M_TR), 0, exclude=EXCL, athena_train=ATH_TR)
changed = sh('git status --porcelain --untracked-files=no -- configs data/youcookii/annotations', ATHENA)
assert not changed, f'2 fold sinh lại khác file đã commit: {changed}'
FA, FB = set(FOLD['A']), set(FOLD['B'])
assert len(FA) == len(FB) == 501 and FA | FB == M_TR == set(ATH_TR) and not FA & FB and not (FA | FB) & EXCL

# (3) train từng fold (bỏ qua nếu checkpoint đã có trên Drive), export best_cap
FOLD_SRC = {}
for X in ('A', 'B'):
    run_name, keep = f'yc2clean_fold{X}', (FA if X == 'A' else FB)
    src_path, out_run = f'{FOLD_DIR}/fold{X}.json', f'{FOLD_DIR}/run_yc2clean_fold{X}'
    if os.path.exists(src_path):
        s = json.load(open(src_path))
        if os.path.exists(s['checkpoint']) and sha256(s['checkpoint']) == s['sha256']:
            FOLD_SRC[X] = s
            print(f'fold {X}: đã có {s["sha256"][:16]} ({s["checkpoint"]})')
            continue
    run_dir, rec_dst = f'{ATHENA}/ckpt/{run_name}', f'{ATHENA}/experiments/runs/colab-{run_name}.json'
    rec_ok = os.path.exists(f'{out_run}/record.json') and json.load(open(f'{out_run}/record.json')).get('finished')
    if rec_ok and os.path.exists(f'{out_run}/best_cap.pth.tar'):
        os.makedirs(run_dir, exist_ok=True); os.makedirs(os.path.dirname(rec_dst), exist_ok=True)
        shutil.copy(f'{out_run}/best_cap.pth.tar', f'{run_dir}/best_cap.pth.tar')
        shutil.copy(f'{out_run}/record.json', rec_dst)
        print(f'fold {X}: train xong từ trước, đọc lại từ {out_run}')
    else:
        shutil.rmtree(run_dir, ignore_errors=True)
        for p in glob.glob(f'{ATHENA}/experiments/runs/*-{run_name}.json'):
            os.remove(p)
        cmd = [sys.executable, '-u', 'train.py', FOLD['files'][X]['config'], '--output', run_name, '--note', f'UEMR 2 fold {X}',
               '--set', 'num_workers=4', 'train_cfg.ema_decay=0.999']
        print(' '.join(cmd), flush=True)
        t1 = time.time()
        with open(f'/content/train_{X}.log', 'w') as lf:
            p = subprocess.Popen(cmd, cwd=ATHENA, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
            for line in p.stdout:
                print(line, end='', flush=True); lf.write(line)
        assert p.wait() == 0, f'train.py fold {X} lỗi: xem log ở trên'
        print(f'fold {X}: train {(time.time() - t1) / 60:.0f} phút')
        os.makedirs(out_run, exist_ok=True)
        shutil.copy(f'/content/train_{X}.log', f'{out_run}/train.log')
        shutil.copy(f'{run_dir}/best_cap.pth.tar', f'{out_run}/best_cap.pth.tar')
        shutil.copy(glob.glob(f'{ATHENA}/experiments/runs/*-{run_name}.json')[0], f'{out_run}/record.json')
    rec, log = json.load(open(f'{out_run}/record.json')), open(f'{out_run}/train.log').read()
    c = rec['config']
    assert 'Data         : 501 train / 104 val videos' in log, f'fold {X}: log không in 501 train / 104 val videos'
    assert c['dataset']['json_file'].endswith(f'yc2clean_fold{X}.json') and c['train_split'] == ['training'] and c['val_split'] == ['dev']
    assert c['dataset']['caption_emb_file'].endswith('_yc2clean.npz') and c['train_cfg']['ema_decay'] == 0.999 and c['num_workers'] == 4
    assert rec['git']['dirty'] is False and rec['finished'] and len(rec['history']) == 12, (rec['git'], len(rec['history']))
    api = f'{ATHENA}/ckpt/api/athena_{run_name}.pth'
    if not os.path.exists(api):
        sh(f'{sys.executable} tools/export_api_ckpt.py {run_name} --no-seg --out {api}', ATHENA)
    ck = torch.load(api, map_location='cpu', weights_only=False)
    assert ck['run'] == run_name and 'state_dict_seg' not in ck and ck['epoch'] == rec['best']['CIDEr']['epoch']
    sha = sha256(api)
    dst = f'{FOLD_DIR}/{sha[:8]}/athena_{run_name}.pth'
    os.makedirs(os.path.dirname(dst), exist_ok=True)
    if not (os.path.exists(dst) and sha256(dst) == sha):
        shutil.copy(api, dst)
    s = {'checkpoint': dst, 'sha256': sha, 'run': run_name, 'fold': X, 'q_for_videos_of': 'B' if X == 'A' else 'A',
         'train_videos': sorted(keep), 'epoch (best_cap, CIDEr dev)': ck['epoch'], 'best': rec['best'], 'git': rec['git'],
         'argv': rec['argv'], 'annotation': FOLD['files'][X]['annotation'],
         'annotation_sha256': sha256(f'{ATHENA}/{FOLD["files"][X]["annotation"]}'), 'caption_emb_sha256': CAP_SHA,
         'created': time.strftime('%Y-%m-%d %H:%M:%S')}
    for p in (f'{FOLD_DIR}/{sha[:8]}/SOURCE.json', src_path):
        json.dump(s, open(p, 'w'), indent=1, ensure_ascii=False)
    FOLD_SRC[X] = s
SA_, SB_ = set(FOLD_SRC['A']['train_videos']), set(FOLD_SRC['B']['train_videos'])
FOLD_CHECK = {'A': len(SA_), 'B': len(SB_), 'A ∪ B == trainsplit 1002': SA_ | SB_ == M_TR, 'A ∩ B': len(SA_ & SB_),
              '(A ∪ B) ∩ (dev ∪ val)': len((SA_ | SB_) & EXCL), 'sha256': {X: FOLD_SRC[X]['sha256'] for X in 'AB'},
              'epoch best_cap': {X: FOLD_SRC[X]['epoch (best_cap, CIDEr dev)'] for X in 'AB'}}
print(json.dumps(FOLD_CHECK, indent=1, ensure_ascii=False))
check('7.4.5 hai fold rời nhau, hợp = 1.002', FOLD_CHECK)
assert FOLD_CHECK['A ∪ B == trainsplit 1002'] and FOLD_CHECK['A ∩ B'] == FOLD_CHECK['(A ∪ B) ∩ (dev ∪ val)'] == 0
print(f'cell 5: {(time.time() - t0) / 60:.1f} phút')

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]

fold A: đã có ec46b25345d9bcee (/content/drive/MyDrive/Colab Notebooks/code KL/Fushion/uemr_2/athena_folds/ec46b253/athena_yc2clean_foldA.pth)
fold B: đã có dc479e0304c843a5 (/content/drive/MyDrive/Colab Notebooks/code KL/Fushion/uemr_2/athena_folds/dc479e03/athena_yc2clean_foldB.pth)
{
 "A": 501,
 "B": 501,
 "A ∪ B == trainsplit 1002": true,
 "A ∩ B": 0,
 "(A ∪ B) ∩ (dev ∪ val)": 0,
 "sha256": {
  "A": "ec46b25345d9bcee1c5c70efed28e9fdc80a28f794a7999ca35b2239851ef92d",
  "B": "dc479e0304c843a5526580a9081906828f04d9beb01ca57b5ae4adb6769ab174"
 },
 "epoch best_cap": {
  "A": 7,
  "B": 10
 }
}
cell 5: 1.2 phút


In [7]:
# ===== 6. q (đặc tả 3.2): val/dev mọi dòng + GT từ 961f1a89; train (GT, R6) từ fold không thấy video, và từ 961f1a89 (A1) =====
from uemr2 import athena_vec as AV
from uniav_app.features import get_source
t0 = time.time()

def cached(rel):
    """Bản local của một file cache UEMR 2 (chép từ Drive nếu có); push(rel) chép ngược lên Drive."""
    loc, drv = f'{LOCAL}/{rel}', f'{U2}/{rel}'
    if not os.path.exists(loc) and os.path.exists(drv):
        os.makedirs(os.path.dirname(loc), exist_ok=True); shutil.copy2(drv, loc)
    return loc

def push(rel):
    os.makedirs(os.path.dirname(f'{U2}/{rel}'), exist_ok=True)
    shutil.copy2(f'{LOCAL}/{rel}', f'{U2}/{rel}')

def local_ckpt(path, sha):
    dst = f'/content/ckpt/{os.path.basename(path)}'
    os.makedirs('/content/ckpt', exist_ok=True)
    if not (os.path.exists(dst) and sha256(dst) == sha):
        shutil.copy2(path, dst)
    assert sha256(dst) == sha, path
    return dst

CK = local_ckpt(ATHENA_CKPT, ATHENA_SHA)
IV2_ENC = f'{ATHENA}/ckpt/internvideo2/InternVideo2-stage2_1b-224p-f4.pt'     # tháp chữ IV2 cho u (cell 7)
CFG_KW = dict(feature_source='drive', drive_feats_src=FEATS, cache_root=f'{LOCAL}/uniav', omni_subset=SUBSET,
              iv2_video_encoder=IV2_ENC)
DUR = {r['video_id']: float(r['duration']) for r in json.load(open(SUBSET))['videos']}
DUR.update({v: float(d) for v, d in json.load(open(f'{UC}/ft_train/durations.json')).items() if v in M_TR})
pipe, cfgA = AV.load(CK, ATHENA_RUN, **CFG_KW)
assert pipe.seg_model is pipe.model                            # q dùng state_dict (best_cap); đoạn đã có sẵn
REC = get_source(cfgA).prepare(sorted(GAL | M_TR), {v: DUR[v] for v in GAL | M_TR})
assert len(REC) == len(GAL | M_TR) == 1500, f'thiếu feature: {len(REC)} / 1500'   # 498 gallery + 1.002 train

SEG_GAL = AV.segments_by_video(*(G[s, row] for s in ('val', 'dev') for row in D.ROWS))
SEG_TR = AV.segments_by_video(TR)
QREL = {'gallery': f'vec/q/{ATHENA_H8}/gallery.npz', 'train': f'vec/q/{ATHENA_H8}/train.npz'}
QG = AV.q_cache(cached(QREL['gallery']), pipe, REC, SEG_GAL); push(QREL['gallery'])
Q961 = AV.q_cache(cached(QREL['train']), pipe, REC, SEG_TR); push(QREL['train'])

# kiểm 7.4.3: q tự tính cho đoạn GT val = embeds_gt của inference trong repo train Athena (20 video)
V20 = sorted(VIDS['val'])[:20]
REPO_GT = AV.repo_gt_vectors(ATHENA, CK, 'validation', V20, CAP_LOCAL)
mine = AV.q_vectors(pipe, REC, {v: {f'{v}#{j}': tuple(map(float, s)) for j, s in enumerate(REPO_GT[v][0])} for v in V20})
cos743 = np.array([float(mine[f'{v}#{j}'] @ e) for v in V20 for j, e in enumerate(REPO_GT[v][1])])
print(f'7.4.3: {len(cos743)} đoạn GT của 20 video val, cos min {cos743.min():.6f}, trung vị {np.median(cos743):.6f}')
check('7.4.3 q == embeds_gt (cos min, 20 video val)', float(cos743.min()))
assert cos743.min() > 0.999, 'DỪNG (7.4.3): span_vectors khác embeds_gt của Athena'

# train: fold X cho video của nửa kia
QF = {}
for X, other in (('A', FB), ('B', FA)):
    s = FOLD_SRC[X]
    pf, _ = AV.load(local_ckpt(s['checkpoint'], s['sha256']), s['run'], **CFG_KW)
    rel = f'vec/q/{s["sha256"][:8]}/train_{"B" if X == "A" else "A"}.npz'
    QREL[f'fold{X}'] = rel
    QF[X] = AV.q_cache(cached(rel), pf, REC, {v: SEG_TR[v] for v in other}); push(rel)
    del pf; torch.cuda.empty_cache()
n_tr = {X: sum(len(SEG_TR[v]) for v in other) for X, other in (('A', FB), ('B', FA))}
print('q:', {'gallery': len(QG), 'train 961f1a89': len(Q961), 'fold A -> B': len(QF['A']), 'fold B -> A': len(QF['B'])}, '| cần', n_tr)
assert set(QG) >= set(k for g in G.values() for k in g['seg_key'].tolist()) and set(Q961) >= set(TR['seg_key'].tolist())
assert all(set(QF[X]) >= {k for v in other for k in SEG_TR[v]} for X, other in (('A', FB), ('B', FA)))
print(f'cell 6: {(time.time() - t0) / 60:.1f} phút')

[drive] 1500 videos: 0 ready, 1500 to prepare -> /content/u2/uniav/feats/drive
[q] /content/u2/vec/q/961f1a89/gallery.npz: 0 cached, 498/498 videos to run
[q] 200/498 videos, 12289 segments
[q] 400/498 videos, 24408 segments
[q] 498/498 videos, 30374 segments
[q] /content/u2/vec/q/961f1a89/train.npz: 0 cached, 1002/1002 videos to run
[q] 200/1002 videos, 3183 segments
[q] 400/1002 videos, 6319 segments
[q] 600/1002 videos, 9454 segments
[q] 800/1002 videos, 12555 segments
[q] 1000/1002 videos, 15542 segments
[q] 1002/1002 videos, 15589 segments
[q] 20/20 videos, 161 segments
7.4.3: 161 đoạn GT của 20 video val, cos min 1.000000, trung vị 1.000000
[q] /content/u2/vec/q/ec46b253/train_B.npz: 0 cached, 501/501 videos to run
[q] 200/501 videos, 3119 segments
[q] 400/501 videos, 6292 segments
[q] 501/501 videos, 7833 segments
[q] /content/u2/vec/q/dc479e03/train_A.npz: 0 cached, 501/501 videos to run
[q] 200/501 videos, 3198 segments
[q] 400/501 videos, 6258 segments
[q] 501/501 videos, 775

In [8]:
# ===== 7. u (đặc tả 3.3) cho câu val, dev; kiểm 7.4.4 =====
from huggingface_hub import hf_hub_download
if not os.path.exists(IV2_ENC):
    hf_hub_download('nguyenminh04/athena', 'internvideo2/InternVideo2-stage2_1b-224p-f4.pt', local_dir=f'{ATHENA}/ckpt')
assert os.path.exists(IV2_ENC)
ck961 = torch.load(CK, map_location='cpu', weights_only=False)
MU = np.asarray(ck961['caption_pool']['mean'], np.float32)
assert ck961['caption_pool']['file'] == 'caption_emb_iv2j_yc2clean.npz', ck961['caption_pool']['file']
assert np.allclose(pipe.captioner.mean.cpu().numpy(), MU), 'pipeline không dùng mean của 961f1a89'
UU, UREL = {}, {}
for s in ('val', 'dev'):
    UREL[s] = f'vec/u/{ATHENA_H8}/{s}.npz'
    have = AV.read(cached(UREL[s]))
    if set(have) != set(Q[s]['caption_id'].tolist()):
        have = dict(zip(Q[s]['caption_id'].tolist(), AV.u_vectors(pipe, Q[s]['sentence'].tolist())))
        AV.save(f'{LOCAL}/{UREL[s]}', have); push(UREL[s])
    UU[s] = D.take(have, Q[s]['caption_id'].tolist())
UU['all'] = UU['val']
cos744 = np.array([float(UU['val'][j] @ pipe.embed_query(str(Q['val']['sentence'][j])).float().cpu().numpy()) for j in range(20)])
print(f'7.4.4: cos(u, pipeline.embed_query) trên 20 câu val: min {cos744.min():.6f} | |mean| {np.linalg.norm(MU):.4f}')
check('7.4.4 u == embed_query (cos min, 20 câu), mean 961f1a89, clip_proj', float(cos744.min()))
assert cos744.min() > 0.999 and UU['val'].shape == (3030, 512) and UU['dev'].shape == (773, 512)

internvideo2/InternVideo2-stage2_1b-224p(…): reconstructing file:   0%|          |  0.00B / 2.82GB            

internvideo2/InternVideo2-stage2_1b-224p(…): downloading bytes:           |  0.00B            

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

7.4.4: cos(u, pipeline.embed_query) trên 20 câu val: min 0.999936 | |mean| 0.9714


## P_φ (đặc tả 4)

In [9]:
# ===== 8. Dữ liệu train P_φ (đặc tả 4.3): câu của từng đoạn, pool câu duy nhất, vector IV2 của pool =====
IV2 = D.load_iv2(CAP_LOCAL)
TS = D.train_set(TR, QT, IV2)
print(json.dumps(TS['counts'], indent=1, ensure_ascii=False))
assert TS['counts']['gt rows'] == 7787 and TS['counts']['r6 rows'] == 7802
assert TS['counts']['unique captions (pool)'] == len(set(QT['sentence'].tolist())) == len(TS['T_pool']) == len(TS['V_pool'])
Q_TR = {'961': D.take(Q961, TR['seg_key'].tolist()),
        'cf': np.stack([(QF['A'] if v in FB else QF['B'])[k] for k, v in zip(TR['seg_key'].tolist(), TR['video'].tolist())])}
print('cos(q CF2, q 961f1a89) trên đoạn train: trung bình %.4f' % float((Q_TR['cf'] * Q_TR['961']).sum(1).mean()))

def train_data(q, no_fn=False):
    """TrainData của uemr2_core. no_fn (A4): mỗi đoạn một 'video' riêng, nên false_negative_pairs rỗng (F(i) = ∅);
    vid không dùng vào việc gì khác trong uemr2_core."""
    vid = np.arange(len(q)) if no_fn else TS['vid']
    return U.TrainData(q=torch.from_numpy(q), e=torch.from_numpy(TR['E']), cap=torch.from_numpy(TS['cap']),
                       vid=torch.from_numpy(vid), segs=torch.from_numpy(TR['segs']),
                       txt_pool=torch.from_numpy(TS['T_pool']), iv2_pool=torch.from_numpy(TS['V_pool']))
check('dữ liệu train P_φ', TS['counts'])
assert Q_TR['cf'].shape == Q_TR['961'].shape == (len(TR['seg_key']), 512)

{
 "rows": 15589,
 "gt rows": 7787,
 "r6 rows": 7802,
 "r6 with caption": 5255,
 "r6 without caption": 2547,
 "captions": 7787,
 "unique captions (pool)": 7509
}
cos(q CF2, q 961f1a89) trên đoạn train: trung bình 0.7252


In [10]:
# ===== 9. Train P_φ: (U) = CF2, A1 noCF, A2 λ=0, A3 không L_av, A4 F=∅; seed 0/1/2; tiêu chí dev = R@1 dev R8 với S^A =====
VARIANTS = {'U': {'q': 'cf'}, 'A1': {'q': '961'}, 'A2': {'q': 'cf', 'lam_txt': 0.0},
            'A3': {'q': 'cf', 'no_av': True}, 'A4': {'q': 'cf', 'no_fn': True}}
HYPER = {'epochs': 30, 'batch': 512, 'lr': 1e-3, 'wd': 1e-4, 'patience': 5, 'alpha': 0.5, 'tau_soft': 0.02}   # đặc tả 4.5
K8 = ('dev', 'R8')
Q_DEV8 = torch.from_numpy(D.take(QG, G[K8]['seg_key'].tolist())).cuda()

def dev_eval(model):
    h = U.align(model, Q_DEV8).cpu().numpy()
    return run(K8, [(1.0, D.scores(Q['dev']['T'], h))])['R@1']

PROJ = {}
for name in RUNS:
    v = VARIANTS[name]
    for s in SEEDS:
        rel = f'proj/{name}/seed{s}.pt'
        if os.path.exists(cached(rel)):
            PROJ[name, s] = torch.load(f'{LOCAL}/{rel}', map_location='cpu', weights_only=False)
            continue
        t1 = time.time()
        orig = U.loss_av
        if v.get('no_av'):                                       # A3: L = L_txt (λ = 1), L_av bỏ hẳn
            U.loss_av = lambda h, rows, bank, scale, neg_mask=None: h.sum() * 0
        try:
            model, info = U.train_projector(train_data(Q_TR[v['q']], v.get('no_fn', False)), seed=s,
                                            lam_txt=v.get('lam_txt', 1.0), dev_eval=dev_eval, device='cuda',
                                            log=lambda r: print(f'  {name} seed {s} ' + ' '.join(f'{k} {x:.4f}' if isinstance(x, float) else f'{k} {x}'
                                                                                              for k, x in r.items()), flush=True),
                                            **{k: HYPER[k] for k in ('epochs', 'batch', 'lr', 'wd', 'patience', 'alpha', 'tau_soft')})
        finally:
            U.loss_av = orig
        PROJ[name, s] = {'state_dict': model.state_dict(), 'info': info, 'variant': v, 'seed': s, 'hyper': HYPER}
        os.makedirs(os.path.dirname(f'{LOCAL}/{rel}'), exist_ok=True)
        torch.save(PROJ[name, s], f'{LOCAL}/{rel}'); push(rel)
        print(f'{name} seed {s}: epoch tốt nhất {info["best_epoch"]}, R@1 dev (S^A) {info["best_dev"]:.2f} | {time.time() - t1:.0f} s')
for name in RUNS:
    hist = {s: PROJ[name, s]['info']['history'] for s in SEEDS}
    json.dump(hist, open(f'{LOCAL}/proj_{name}_history.json', 'w'), indent=1)
    os.makedirs(f'{U2}/proj/{name}', exist_ok=True); shutil.copy2(f'{LOCAL}/proj_{name}_history.json', f'{U2}/proj/{name}/history.json')
def best_row(name, s):
    i = PROJ[name, s]['info']
    return next(r for r in i['history'] if r['epoch'] == i['best_epoch'])
TRAIN_TAB = pd.DataFrame([{'run': n, 'seed': s, 'epoch tốt nhất': PROJ[n, s]['info']['best_epoch'],
                           'epoch chạy': len(PROJ[n, s]['info']['history']), 'R@1 dev S^A': PROJ[n, s]['info']['best_dev'],
                           'L_av': best_row(n, s)['L_av'], 'L_txt': best_row(n, s)['L_txt'],
                           '1/σ_av': best_row(n, s)['tau_av'], '1/σ_txt': best_row(n, s)['tau_txt']} for n in RUNS for s in SEEDS])
display(TRAIN_TAB.round(4))
assert all((n, s) in PROJ for n in RUNS for s in SEEDS)

  U seed 0 epoch 1 L_av 6.6968 L_txt 6.3999 dev 27.4256 tau_av 0.0680 tau_txt 0.0680
  U seed 0 epoch 2 L_av 5.8379 L_txt 5.8082 dev 30.7891 tau_av 0.0657 tau_txt 0.0658
  U seed 0 epoch 3 L_av 5.6110 L_txt 5.6745 dev 30.4010 tau_av 0.0635 tau_txt 0.0637
  U seed 0 epoch 4 L_av 5.4356 L_txt 5.5624 dev 30.6598 tau_av 0.0615 tau_txt 0.0618
  U seed 0 epoch 5 L_av 5.2825 L_txt 5.4621 dev 30.9185 tau_av 0.0596 tau_txt 0.0599
  U seed 0 epoch 6 L_av 5.1467 L_txt 5.3686 dev 30.5304 tau_av 0.0578 tau_txt 0.0582
  U seed 0 epoch 7 L_av 5.0186 L_txt 5.2768 dev 30.5304 tau_av 0.0561 tau_txt 0.0565
  U seed 0 epoch 8 L_av 4.9019 L_txt 5.2005 dev 29.6248 tau_av 0.0545 tau_txt 0.0550
  U seed 0 epoch 9 L_av 4.7964 L_txt 5.1259 dev 30.5304 tau_av 0.0531 tau_txt 0.0536
  U seed 0 epoch 10 L_av 4.6924 L_txt 5.0526 dev 29.1074 tau_av 0.0517 tau_txt 0.0523
U seed 0: epoch tốt nhất 5, R@1 dev (S^A) 30.92 | 5 s
  U seed 1 epoch 1 L_av 6.6996 L_txt 6.3968 dev 27.6843 tau_av 0.0680 tau_txt 0.0680
  U seed 1

,run,seed,epoch tốt nhất,epoch chạy,R@1 dev S^A,L_av,L_txt,1/σ_av,1/σ_txt
0,U,0,5,10,30.9185,5.2825,5.4621,0.0596,0.0599
1,U,1,4,9,31.6947,5.4381,5.5651,0.0615,0.0618
2,U,2,5,10,31.5653,5.2879,5.4667,0.0596,0.0599
3,A1,0,11,16,33.8939,4.1521,4.0007,0.0508,0.0515
4,A1,1,5,10,34.6701,4.8384,4.4666,0.0597,0.0600
5,A1,2,7,12,33.7646,4.5775,4.2906,0.0563,0.0567
6,A2,0,3,8,26.7788,5.3174,6.8015,0.0635,0.0700
7,A2,1,3,8,26.3907,5.3222,6.8036,0.0635,0.0700
8,A2,2,3,8,27.6843,5.3169,6.8029,0.0635,0.0700
9,A3,0,4,9,30.4010,0.0000,5.2828,0.0700,0.0618


In [11]:
# ===== 10. h̄ (trung bình 3 seed, không chuẩn hoá lại, lưu fp16) cho mọi đoạn val/dev; năng lực text -> clip GT trên dev =====
from omni_retrieval.evaluate import text_to_clip as omni_text_to_clip
KEYS_GAL = sorted(set(k for g in G.values() for k in g['seg_key'].tolist()))
Q_GAL = torch.from_numpy(D.take(QG, KEYS_GAL)).cuda()
H, HREL = {}, {}
for name in RUNS:
    HREL[name] = f'vec/h/{name}/gallery.npz'
    have = AV.read(cached(HREL[name]))
    if set(have) != set(KEYS_GAL):
        hs = []
        for s in SEEDS:
            m = U.AlignProjector()
            m.load_state_dict(PROJ[name, s]['state_dict'])
            hs.append(U.align(m.cuda().eval(), Q_GAL).cpu().numpy())
        have = dict(zip(KEYS_GAL, np.mean(hs, 0)))                  # đặc tả 4.5: h̄ = (1/3) Σ_s h^(s)
        AV.save(f'{LOCAL}/{HREL[name]}', have, dtype=np.float16); push(HREL[name])
        have = AV.read(f'{LOCAL}/{HREL[name]}')                     # dùng đúng bản fp16 đã lưu
    H[name] = have

def clip_ability(s):
    """text -> clip GT (một clip mỗi câu, như omni_retrieval.evaluate.text_to_clip 'eval') cho e, h̄ (U), q (với u)."""
    g7 = G[s, 'R7']
    rows = D.clip_rows(Q[s], g7)
    keys = g7['seg_key'][rows].tolist()
    return {'e (Omni ft)': D.text_to_clip(Q[s]['T'], g7['E'][rows]), 'h̄ (U)': D.text_to_clip(Q[s]['T'], D.take(H['U'], keys)),
            'q (với u)': D.text_to_clip(UU[s], D.take(QG, keys))}
CLIP_DEV = clip_ability('dev')
display(pd.DataFrame(CLIP_DEV).T.round(2))
ref = omni_text_to_clip(DB, 'text_dev.npz')['eval']
assert abs(CLIP_DEV['e (Omni ft)']['R@1'] - ref['R@1']) < 1e-9, (CLIP_DEV['e (Omni ft)'], ref)

,R@1,R@5,R@10,MdR,clips
e (Omni ft),56.40,84.22,91.20,1.0,773.0
h̄ (U),29.62,65.33,77.10,3.0,773.0
q (với u),30.53,60.41,72.83,3.0,773.0


In [12]:
# ===== 11. Chọn trên dev: z-score cho (e), lưới β (U) và β_I (e) trên dòng R8; khối configs/uemr2_protocol.json =====
z64 = lambda Z, S: Z(S.astype(np.float64))            # z-score trong float64: β_I = 0 giữ đúng thứ tự (và hoà) của S^Ω
SA = {'U': {k: D.scores(Q[k[0]]['T'], D.take(H['U'], G[k]['seg_key'].tolist())) for k in G if k[0] == 'dev'}}
SI = {k: D.scores(UU[k[0]], D.take(QG, G[k]['seg_key'].tolist())) for k in G if k[0] == 'dev'}
ZO, ZI = U.zstats(SOM[K8]), U.zstats(SI[K8])           # đặc tả 5.2: toàn ma trận 773 câu x mọi đoạn R8 dev
g8 = G[K8]
sel = lambda make: U.select_beta(GRID, make, g8['seg_vid'], len(g8['videos']), CV[K8], Q['dev']['gt'], g8['segs'])
BETA, GRID_U = sel(lambda b: [(1.0, SOM[K8]), (b, SA['U'][K8])])
BETA_I, GRID_E = sel(lambda b: [(1.0, z64(ZO, SOM[K8])), (b, z64(ZI, SI[K8]))])
display(pd.DataFrame(GRID_U).round(2).assign(cột='(U)')); display(pd.DataFrame(GRID_E).round(2).assign(cột='(e)'))
print(f'β* = {BETA} | β_I* = {BETA_I}')
r0 = run(K8, [(1.0, SOM[K8]), (0.0, SA['U'][K8])])
e0 = run(K8, [(1.0, z64(ZO, SOM[K8])), (0.0, z64(ZI, SI[K8]))])
same_dev = bool(np.array_equal(r0['rank'], REFR[K8]['rank']) and np.array_equal(e0['rank'], REFR[K8]['rank']))
check('7.4.2 β=0 (U) và β_I=0 (e) = hạng R8 dev', same_dev)
assert same_dev, 'DỪNG (7.4.2): β = 0 không cho đúng mảng hạng R8 dev'

sha16 = lambda p: sha256(p)[:16]
BLOCK = {
    'version': 1, 'dataset': 'YouCook2', 'spec': 'docs/uemr2/UEMR_ALIGN_FINAL.md (2026-10-23)',
    'checkpoints': {'athena': {'path': ATHENA_CKPT, 'sha256': ATHENA_SHA, 'run': ATHENA_RUN},
                    **{f'fold_{X}': {k: FOLD_SRC[X][k] for k in ('checkpoint', 'sha256', 'run', 'epoch (best_cap, CIDEr dev)', 'annotation_sha256')}
                       for X in 'AB'},
                    'omni_ft': 'checkpoint-1948 (vector có sẵn trong cache UEMR)'},
    'folds': {'seed': 0, 'source': 'train_omni_video.trainsplit.jsonl', 'manifest_sha256': TRAIN_MANIFEST_SHA,
              'A': len(FA), 'B': len(FB), 'athena_commit': ATHENA_COMMIT},
    'segments': {'main_row': 'R8', 'uniav_hash8': H8, 'theta': 0.4, 'L_min_s': 5, 'L_fill_s': 10},
    'train': {'hyper': {**HYPER, 'optimizer': 'AdamW, cosine theo bước', 'lambda_txt': 1.0, 'tiou_false_negative': 0.5,
                        'caption_tiou_r6': 0.5, 'precision': 'fp32 train, h̄ fp16'},
              'seeds': list(SEEDS), 'dev_criterion': 'R@1 dev (104 video, 773 câu), đoạn R8, điểm <t, h> (S^A)',
              'data': TS['counts'],
              'runs': {n: {'variant': VARIANTS[n], 'best_epoch': [PROJ[n, s]['info']['best_epoch'] for s in SEEDS],
                           'best_dev_R@1': [round(PROJ[n, s]['info']['best_dev'], 4) for s in SEEDS],
                           'tau_av': [round(best_row(n, s)['tau_av'], 5) for s in SEEDS],
                           'tau_txt': [round(best_row(n, s)['tau_txt'], 5) for s in SEEDS]} for n in RUNS}},
    'beta_grid': GRID, 'beta_star': BETA, 'beta_I_star': BETA_I,
    'dev_grid_U': [{k: (round(v, 4) if isinstance(v, float) else v) for k, v in r.items()} for r in GRID_U],
    'dev_grid_e': [{k: (round(v, 4) if isinstance(v, float) else v) for k, v in r.items()} for r in GRID_E],
    'zstats_dev_R8': {'omega': {'mu': ZO.mu, 'sd': ZO.sd}, 'iv2': {'mu': ZI.mu, 'sd': ZI.sd}},
    'rules': {'beta': 'R@1 dev cao nhất trên dòng R8, hoà lấy β nhỏ hơn; áp nguyên cho mọi dòng',
              'val': 'gallery val 394, 3.030 câu, chấm một lần; kho 498 một dòng phụ',
              'bootstrap': {'unit': 'video', 'n_boot': 1000, 'seed': 0, 'ci': 95, 'code': 'uemr2_core.paired_bootstrap'},
              'criteria': {'1': 'lo CI (U) - R8 (R@1 val 394) > 0 -> UEMR 2 là phương pháp chính YC2',
                           '2': 'lo CI (U) - (e) trên R8 > 0 -> căn chỉnh tốt hơn gộp thẳng hai không gian',
                           '3': 'lo CI C1 - R6 > 0 -> căn chỉnh thay được Omni ở đoạn lấp',
                           '4': 'VCMR (U) - R8 kèm CI, không ngưỡng',
                           '5': 'β* = 0 -> dừng nhánh (U), báo cáo val với β = 0, vẫn chạy (h) và C1',
                           '6': 'kiểm tra 7.4 không qua -> dừng'}},
    'cache': {**{f'q {k}': sha16(f'{LOCAL}/{r}') for k, r in QREL.items()}, **{f'u {k}': sha16(f'{LOCAL}/{r}') for k, r in UREL.items()},
              **{f'h̄ {n}': sha16(f'{LOCAL}/{r}') for n, r in HREL.items()},
              **{f'P_phi {n} seed{s}': sha16(f'{LOCAL}/proj/{n}/seed{s}.pt') for n in RUNS for s in SEEDS}},
    'clip_dev': {k: {m: round(x, 4) for m, x in v.items()} for k, v in CLIP_DEV.items()},
    'code': {'fushion_commit_dev': FUSHION_COMMIT, 'athena_commit': ATHENA_COMMIT},
}
if PHASE == 'dev':
    json.dump(BLOCK, open(f'{U2}/protocol_dev.json', 'w'), indent=1, ensure_ascii=False)
    print(json.dumps(BLOCK, indent=1, ensure_ascii=False))
    print('-> gửi log (hoặc file', f'{U2}/protocol_dev.json', ') cho agent: ghi configs/uemr2_protocol.json, commit, rồi lượt val')
assert BETA in GRID and BETA_I in GRID

,beta,R@1,R@5,R@10,MdR,MnR,VCMR@1,mom@.5|top1,cột
0,0.00,46.31,81.63,91.33,2.0,4.19,24.58,53.07,(U)
1,0.10,48.12,83.18,91.33,2.0,3.86,26.26,54.57,(U)
2,0.20,46.96,82.41,91.20,2.0,3.94,25.49,54.27,(U)
3,0.30,45.67,81.63,91.20,2.0,4.15,25.10,54.96,(U)
4,0.50,45.02,81.11,90.30,2.0,4.56,25.74,57.18,(U)
5,0.75,42.95,79.95,89.65,2.0,4.90,24.06,56.02,(U)
6,1.00,41.91,78.91,88.62,2.0,5.18,23.03,54.94,(U)


,beta,R@1,R@5,R@10,MdR,MnR,VCMR@1,mom@.5|top1,cột
0,0.00,46.31,81.63,91.33,2.0,4.19,24.58,53.07,(e)
1,0.10,47.99,83.05,91.20,2.0,3.95,26.00,54.18,(e)
2,0.20,49.16,82.54,91.46,2.0,3.81,26.26,53.42,(e)
3,0.30,49.68,82.92,91.59,2.0,3.77,26.78,53.91,(e)
4,0.50,48.25,82.54,92.11,2.0,3.80,26.00,53.89,(e)
5,0.75,47.61,81.89,91.72,2.0,3.96,25.74,54.08,(e)
6,1.00,45.80,80.60,90.30,2.0,4.18,25.23,55.08,(e)


β* = 0.1 | β_I* = 0.3
{
 "version": 1,
 "dataset": "YouCook2",
 "spec": "docs/uemr2/UEMR_ALIGN_FINAL.md (2026-10-23)",
 "checkpoints": {
  "athena": {
   "path": "/content/drive/MyDrive/Colab Notebooks/code KL/Fushion/uemr/athena_sach/961f1a89/athena_yc2clean.pth",
   "sha256": "961f1a895f4b43c811d3e0cfa31448641989a0a0e39e421ed397682407643d58",
   "run": "yc2clean_E1"
  },
  "fold_A": {
   "checkpoint": "/content/drive/MyDrive/Colab Notebooks/code KL/Fushion/uemr_2/athena_folds/ec46b253/athena_yc2clean_foldA.pth",
   "sha256": "ec46b25345d9bcee1c5c70efed28e9fdc80a28f794a7999ca35b2239851ef92d",
   "run": "yc2clean_foldA",
   "epoch (best_cap, CIDEr dev)": 7,
   "annotation_sha256": "2090387cfb1382cc7cfce552f156d573bfc01bc901b0a0df826cc28ddad5ceee"
  },
  "fold_B": {
   "checkpoint": "/content/drive/MyDrive/Colab Notebooks/code KL/Fushion/uemr_2/athena_folds/dc479e03/athena_yc2clean_foldB.pth",
   "sha256": "dc479e0304c843a5526580a9081906828f04d9beb01ca57b5ae4adb6769ab174",
   "run": "yc

## Lượt val (chỉ sau khi `configs/uemr2_protocol.json` đã commit)

Không đảo thứ tự 11 → 12 → 13.

In [13]:
# ===== 12. Cổng khoá: protocol đã commit, đúng hash, đúng lựa chọn của cell 11 (đặc tả 7.4.6) =====
assert PHASE == 'val', 'lượt dev dừng ở cell 11: agent ghi và commit configs/uemr2_protocol.json trước'
P_PATH = f'{FUSHION}/configs/uemr2_protocol.json'
assert os.path.exists(P_PATH), 'configs/uemr2_protocol.json chưa có ở FUSHION_REF'
P_COMMIT = sh('git log -1 --format="%h %ad" --date=short -- configs/uemr2_protocol.json', FUSHION)
assert P_COMMIT, 'configs/uemr2_protocol.json chưa được commit'
assert not sh('git status --porcelain -- configs/uemr2_protocol.json', FUSHION), 'protocol bị sửa sau commit'
P_SHA16 = sha256(P_PATH)[:16]
assert P_SHA16 == PROTO_SHA16, f'hash protocol {P_SHA16} khác PROTO_SHA16 {PROTO_SHA16}'
PROTO2 = json.load(open(P_PATH))
cmp = lambda a, b: json.dumps(a, sort_keys=True, ensure_ascii=False) == json.dumps(b, sort_keys=True, ensure_ascii=False)
diff = [k for k in BLOCK if k != 'code' and not cmp(BLOCK[k], PROTO2.get(k))]
print('protocol:', P_PATH, '| sha256[:16]', P_SHA16, '| commit', P_COMMIT, '| β*', PROTO2['beta_star'], '| β_I*', PROTO2['beta_I_star'])
check('7.4.6 protocol commit trước val', {'sha256_16': P_SHA16, 'commit': P_COMMIT, 'khác cell 11': diff})
assert not diff, f'DỪNG: cell 11 lượt val khác protocol đã commit ở {diff}'
BETA, BETA_I = PROTO2['beta_star'], PROTO2['beta_I_star']
ZO, ZI = U.ZStats(**PROTO2['zstats_dev_R8']['omega']), U.ZStats(**PROTO2['zstats_dev_R8']['iv2'])
GATE = P_SHA16

AssertionError: lượt dev dừng ở cell 11: agent ghi và commit configs/uemr2_protocol.json trước

In [ ]:
# ===== 13. Chấm val MỘT LẦN: bảng 7.1, kho 498, A1-A4, C1 (bảng 7.2), năng lực text -> clip GT val =====
assert PHASE == 'val' and GATE == PROTO_SHA16
print('protocol', GATE, '| β*', BETA, '| β_I*', BETA_I)
t0 = time.time()
VK = [k for k in G if k[0] in ('val', 'all')]
for n in RUNS:
    SA.setdefault(n, {})
    for k in (VK if n == 'U' else [('val', 'R8')]):
        SA[n][k] = D.scores(Q[k[0]]['T'], D.take(H[n], G[k]['seg_key'].tolist()))
for k in VK:
    SI[k] = D.scores(UU[k[0]], D.take(QG, G[k]['seg_key'].tolist()))

def parts(col, k, n='U'):
    return {'(b)': lambda: [(1.0, SOM[k])], '(d)': lambda: [(1.0, SI[k])], '(h)': lambda: [(1.0, SA[n][k])],
            '(e)': lambda: [(1.0, z64(ZO, SOM[k])), (BETA_I, z64(ZI, SI[k]))],
            '(U)': lambda: [(1.0, SOM[k]), (BETA, SA[n][k])]}[col]()

COLS = ('(b)', '(d)', '(h)', '(e)', '(U)')
R = {}
for row in D.ROWS:
    for col in COLS:
        R[col, row] = run(('val', row), parts(col, ('val', row)))
for col in ('(b)', '(U)'):
    R[col, 'R8 kho 498'] = run(('all', 'R8'), parts(col, ('all', 'R8')))
for n in RUNS[1:]:
    R[n, 'R8'] = run(('val', 'R8'), parts('(U)', ('val', 'R8'), n))
kv8 = ('val', 'R8')
R['C1', 'R8'] = run(kv8, [(1.0, np.where(G[kv8]['is_fill'][None], SA['U'][kv8], SOM[kv8]))])   # đoạn R6: S^Ω, đoạn lấp: S^A

# kiểm 7.4.2 trên val: β = 0, β_I = 0 cho đúng mảng hạng R8; cột (b) = thứ hạng UEMR
same_val = (np.array_equal(run(kv8, [(1.0, SOM[kv8]), (0.0, SA['U'][kv8])])['rank'], REFR[kv8]['rank'])
            and np.array_equal(run(kv8, [(1.0, z64(ZO, SOM[kv8])), (0.0, z64(ZI, SI[kv8]))])['rank'], REFR[kv8]['rank'])
            and all(np.array_equal(R['(b)', row]['rank'], REFR['val', row]['rank']) for row in D.ROWS))
check('7.4.2 β=0 (U) và β_I=0 (e) = hạng R8 val; (b) = UEMR', bool(same_val))
assert same_val, 'DỪNG (7.4.2) trên val'

MET = {k: metrics(r) for k, r in R.items()}
VPV = {row: len(G['val', row]['seg_key']) / 394 for row in D.ROWS}
TABLES = {}
for m in ('R@1', 'R@5', 'R@10', 'MdR', 'MnR', 'VCMR@1'):
    TABLES[m] = pd.DataFrame([{'dòng': row, 'vec/video': VPV[row], **{col: MET[col, row][m] for col in COLS}} for row in D.ROWS])
    print(f'=== bảng 7.1: {m} (val 394, 3.030 câu) ==='); display(TABLES[m].round(2))
T72 = pd.DataFrame([{'mã': n, **{m: MET[n, 'R8'][m] for m in ('R@1', 'R@5', 'R@10', 'VCMR@1')}} for n in ('(U)', *RUNS[1:], 'C1')])
print('=== bảng 7.2 (R8, val 394) ==='); display(T72.round(2))
T498 = pd.DataFrame([{'cột': c, **MET[c, 'R8 kho 498']} for c in ('(b)', '(U)')])
print('=== kho 498 (R8) ==='); display(T498.round(2))
CLIP_VAL = clip_ability('val')
print('=== text -> clip GT val (3.030 clip) ==='); display(pd.DataFrame(CLIP_VAL).T.round(2))
assert round(CLIP_VAL['e (Omni ft)']['R@1'], 2) == 35.64, CLIP_VAL['e (Omni ft)']
np.savez(f'{LOCAL}/ranks_val.npz', caption_id=Q['val']['caption_id'],
         **{f'{c}|{r}|rank': x['rank'] for (c, r), x in R.items()}, **{f'{c}|{r}|vcmr': (x['rank'] <= 1) & x['moment_hit'] for (c, r), x in R.items()})
shutil.copy2(f'{LOCAL}/ranks_val.npz', f'{RES}/ranks_val.npz')
print(f'cell 13: {time.time() - t0:.0f} s')
assert len(R) == len(D.ROWS) * len(COLS) + 2 + len(RUNS) - 1 + 1

In [ ]:
# ===== 14. Bootstrap theo video (1.000, seed 0): các cặp của đặc tả 6 và 7, R@1 và VCMR R@1; nhóm bỏ sót / độ dài =====
VV = Q['val']['video']
r1 = lambda k: 100.0 * (R[k]['rank'] <= 1)
vc = lambda k: 100.0 * ((R[k]['rank'] <= 1) & R[k]['moment_hit'])
def boot(a, b, m=None):
    m = np.ones(len(VV), bool) if m is None else m
    return U.paired_bootstrap(a[m], b[m], VV[m])
PAIRS = {'(U) − R8': (('(U)', 'R8'), ('(b)', 'R8')), '(U) − (e)': (('(U)', 'R8'), ('(e)', 'R8')),
         'C1 − R6': (('C1', 'R8'), ('(b)', 'R6')), 'C1 − R8': (('C1', 'R8'), ('(b)', 'R8')),
         '(U) − R2': (('(U)', 'R8'), ('(b)', 'R2')), '(U) − (R6 ∪ R2)': (('(U)', 'R8'), ('(b)', 'R6∪R2')),
         **{f'{n} − (U)': ((n, 'R8'), ('(U)', 'R8')) for n in RUNS[1:]},
         '(h) − R8': (('(h)', 'R8'), ('(b)', 'R8')),
         'R2 − R8 ở cột (b)': (('(b)', 'R2'), ('(b)', 'R8')), 'R2 − R8 ở cột (U)': (('(U)', 'R2'), ('(U)', 'R8'))}
BOOT = {name: {'R@1': boot(r1(a), r1(b)), 'VCMR@1': boot(vc(a), vc(b))} for name, (a, b) in PAIRS.items()}
fmt = lambda x: f"{x['diff']:+.2f} [{x['lo']:+.2f}, {x['hi']:+.2f}]"
TB = pd.DataFrame([{'cặp': n, 'R@1': fmt(v['R@1']), 'VCMR R@1': fmt(v['VCMR@1'])} for n, v in BOOT.items()])
display(TB)

PF = json.load(open(f'{FUSHION}/configs/final_protocol.json'))
MISSED = D.missed(Q['val'], G['val', 'R6'], PF['eval']['miss_tiou'])
LEN = Q['val']['gt'][:, 1] - Q['val']['gt'][:, 0]
GROUPS = {'bước bị R6 bỏ sót': MISSED, 'không bỏ sót': ~MISSED,
          'GT < 10 s': LEN < 10, 'GT 10–40 s': (LEN >= 10) & (LEN < 40), 'GT ≥ 40 s': LEN >= 40}
assert int(MISSED.sum()) == 823, int(MISSED.sum())
ku, kb = ('(U)', 'R8'), ('(b)', 'R8')
win, lose = (R[ku]['rank'] <= 1) & (R[kb]['rank'] > 1), (R[kb]['rank'] <= 1) & (R[ku]['rank'] > 1)
TG = pd.DataFrame([{'nhóm': g, 'câu': int(m.sum()), 'R8': float(r1(kb)[m].mean()), '(U)': float(r1(ku)[m].mean()),
                    '(U) − R8': fmt(boot(r1(ku), r1(kb), m)), '(U) thắng R8': int((win & m).sum()), 'R8 thắng (U)': int((lose & m).sum())}
                   for g, m in GROUPS.items()])
display(TG.round(2))
assert len(BOOT) == len(PAIRS) and TG['câu'].iloc[0] + TG['câu'].iloc[1] == 3030

In [ ]:
# ===== 15. Tiêu chí ghi trước (đặc tả 8, khoá trong protocol): bảng kết luận =====
REQUIRED = ['7.4.1 cột (b) tái lập', 'uemr2_core == UEMR (hạng + VCMR, mọi dòng, dev/val/498)', '7.4.2 β=0 (U) và β_I=0 (e) = hạng R8 dev',
            '7.4.2 β=0 (U) và β_I=0 (e) = hạng R8 val; (b) = UEMR', '7.4.3 q == embeds_gt (cos min, 20 video val)',
            '7.4.4 u == embed_query (cos min, 20 câu), mean 961f1a89, clip_proj', '7.4.5 dữ liệu: dev, val không giao train',
            '7.4.5 hai fold rời nhau, hợp = 1.002', '7.4.6 protocol commit trước val']
checks_ok = all(k in CHECK for k in REQUIRED)
lo = lambda n, m='R@1': BOOT[n][m]['lo']
CRIT = [
    {'#': 1, 'tiêu chí': 'lo CI (U) − R8 > 0 (R@1, val 394)', 'giá trị': fmt(BOOT['(U) − R8']['R@1']), 'đạt': lo('(U) − R8') > 0,
     'kết luận': 'UEMR 2 là phương pháp chính trên YC2' if lo('(U) − R8') > 0 else 'R8 vẫn là phương pháp chính; căn chỉnh không cải thiện có ý nghĩa'},
    {'#': 2, 'tiêu chí': 'lo CI (U) − (e) > 0 (R8)', 'giá trị': fmt(BOOT['(U) − (e)']['R@1']), 'đạt': lo('(U) − (e)') > 0,
     'kết luận': 'căn chỉnh vào không gian Omni tốt hơn gộp thẳng hai không gian' if lo('(U) − (e)') > 0 else 'không phân biệt được'},
    {'#': 3, 'tiêu chí': 'lo CI C1 − R6 > 0', 'giá trị': fmt(BOOT['C1 − R6']['R@1']), 'đạt': lo('C1 − R6') > 0,
     'kết luận': 'căn chỉnh thay được Omni ở đoạn lấp, cùng chi phí encode với R6' if lo('C1 − R6') > 0 else '(không viết)'},
    {'#': 4, 'tiêu chí': 'VCMR (U) − R8 (không ngưỡng)', 'giá trị': fmt(BOOT['(U) − R8']['VCMR@1']), 'đạt': None, 'kết luận': 'chỉ báo cáo'},
    {'#': 5, 'tiêu chí': 'β* = 0 trên dev', 'giá trị': f'β* = {BETA}', 'đạt': BETA == 0,
     'kết luận': 'dừng nhánh (U): val với β = 0 (= R8), kết luận âm; vẫn báo (h) và C1' if BETA == 0 else '–'},
    {'#': 6, 'tiêu chí': 'kiểm tra 7.4 qua', 'giá trị': f'{sum(k in CHECK for k in REQUIRED)}/{len(REQUIRED)}', 'đạt': checks_ok,
     'kết luận': '–' if checks_ok else 'dừng, không đọc số'},
]
display(pd.DataFrame(CRIT))
missing = [k for k in REQUIRED if k not in CHECK]
assert checks_ok, f'thiếu kiểm tra {missing}'

In [ ]:
# ===== 16. JSON số liệu (results/uemr2/961f1a89/) cho KET_QUA_UEMR_2_NEW_ATHENA.md =====
clean = lambda d: json.loads(json.dumps(d, default=lambda x: x.tolist() if hasattr(x, 'tolist') else str(x)))
OUTJ = {'created': time.strftime('%Y-%m-%d %H:%M:%S'), 'env': ENV, 'fushion_commit': FUSHION_COMMIT, 'athena_commit': ATHENA_COMMIT,
        'protocol': {'path': 'configs/uemr2_protocol.json', 'sha256_16': GATE, 'commit': P_COMMIT},
        'beta_star': BETA, 'beta_I_star': BETA_I,
        'table_7_1': {m: t.to_dict('records') for m, t in TABLES.items()}, 'table_7_2': T72.to_dict('records'),
        'gallery_498': T498.to_dict('records'), 'text_to_clip': {'dev': CLIP_DEV, 'val': CLIP_VAL},
        'pairs': BOOT, 'groups': TG.to_dict('records'), 'criteria': CRIT, 'train': TRAIN_TAB.to_dict('records'),
        'checks': CHECK, 'cells': {'7.1, 7.2, 498, text->clip': 13, 'cặp, nhóm': 14, 'tiêu chí': 15, 'β, z-score, lưới dev': 11}}
OUTJ = clean(OUTJ)
os.makedirs(f'{FUSHION}/results/uemr2/{ATHENA_H8}', exist_ok=True)
for p in (f'{RES}/uemr2_val.json', f'{FUSHION}/results/uemr2/{ATHENA_H8}/uemr2_val.json'):
    json.dump(OUTJ, open(p, 'w'), indent=1, ensure_ascii=False)
print(json.dumps(OUTJ, indent=1, ensure_ascii=False))
print(f'-> {RES}/uemr2_val.json, {RES}/ranks_val.npz | tổng thời gian {(time.time() - T_START) / 60:.0f} phút')
assert os.path.getsize(f'{RES}/uemr2_val.json') > 0